# PrivLLM-Guard walkthrough

Paper: Alghamdi, *An adaptive differential privacy framework for clinical llms…*, Scientific Reports 16:15781 (2026).

Toy dimensions are used so this notebook runs on CPU. Paper scale is `d=768`, `L=12`, `n=512` (§V). Architecture and equations are unchanged.

In [ ]:
import math
import sys
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.utils import load_config
from src.privacy import (
    AdaptiveGradientClipping,
    AdaptiveNoiseCalibrator,
    GaussianMechanism,
    PrivacyBudgetTracker,
    RDPAccountant,
    exponential_mechanism_sample,
    hierarchical_layer_sigmas,
)
from src.model import ModelConfig, PrivacyAwareAttention, PrivacyAwareEmbedding, PrivLLMGuard
from src.loss import LossWeights, combined_loss
from src.evaluate import privacy_score
from src.data import SyntheticClinicalNotes

cfg = load_config(str(ROOT / "configs" / "walkthrough.yaml"))
torch.manual_seed(0)
print("device: cpu")
print("walkthrough d_model", cfg["model"]["d_model"], "vs paper 768")

## (ε,δ)-DP and Gaussian calibration (§IV.A–B, Eqs. 1–2, 6)

> For any two adjacent datasets D and D′ that differ in at most one individual’s data, and for any possible output O, the mechanism M satisfies:
> Pr[M(D)∈O] ≤ e^ε · Pr[M(D′)∈O] + δ
>
> — §IV.A, Eq. 1

**Eq. 6:** $\sigma = C \sqrt{2\ln(1.25/\delta)} / \varepsilon$

In [ ]:
C, eps, delta = 1.2, 0.1, 1e-6  # Appendix A
sigma = GaussianMechanism.calibrate_sigma(C, eps, delta)
sigma_tighter = GaussianMechanism.calibrate_sigma(C, 0.01, delta)
assert sigma_tighter > sigma, "smaller ε must yield larger σ (Eq. 6)"
x = torch.zeros(4, 8)
noisy = GaussianMechanism.add_noise(x, sigma=0.5)
assert noisy.shape == x.shape
assert not torch.allclose(noisy, x)
print(f"✓ Eq. 6: σ(ε=0.1)={sigma:.4f} < σ(ε=0.01)={sigma_tighter:.4f}")

## Embedding perturbation (§IV.A, Eq. 3)

> $\tilde{e}_i = e_i + \mathcal{N}(0, \sigma_{\mathrm{emb}}^2 I)$
>
> Appendix A: $\sigma_{\mathrm{emb}} = 0.5$

In [ ]:
mcfg = ModelConfig.from_dict(cfg)
emb = PrivacyAwareEmbedding(mcfg)
ids = torch.randint(1, mcfg.vocab_size, (2, mcfg.max_seq_len))
clean = emb(ids, sigma_emb=0.0, apply_noise=False)
torch.manual_seed(1)
noisy1 = emb(ids, sigma_emb=mcfg.sigma_emb, apply_noise=True)
torch.manual_seed(1)
noisy2 = emb(ids, sigma_emb=mcfg.sigma_emb, apply_noise=True)
assert clean.shape == (2, mcfg.max_seq_len, mcfg.d_model)
assert not torch.allclose(clean, noisy1)
print(f"✓ Eq. 3 embeddings {tuple(clean.shape)}")

## Privacy-aware attention (§IV.A, Eq. 4)

> $\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\big((QK^T + \mathcal{N}_{\mathrm{att}}) / \sqrt{d_k}\big) V$
>
> Appendix A: $\sigma_{\mathrm{att}} = 0.3$

In [ ]:
attn = PrivacyAwareAttention(mcfg)
h = torch.randn(2, mcfg.max_seq_len, mcfg.d_model)
out_clean = attn(h, h, h, sigma_att=0.0, apply_noise=False)
out_noisy = attn(h, h, h, sigma_att=mcfg.sigma_att, apply_noise=True)
assert out_clean.shape == h.shape
assert not torch.allclose(out_clean, out_noisy)
print(f"✓ Eq. 4 attention {tuple(out_clean.shape)}, d_k={mcfg.d_model // mcfg.n_heads}")

## Adaptive noise calibration (§III Algorithm 1)

> The Adaptive Noise Calibration operates once per sequence, not per token. At sequence start, a single forward pass through the sensitivity analyzer classifies the input’s privacy profile (high/medium/low), determining noise parameters (σ_emb, σ_att) applied uniformly across all tokens.

In [ ]:
anc = AdaptiveNoiseCalibrator(0.5, 0.3, {"high": 1.5, "medium": 1.0, "low": 0.5})
hi = anc.from_profile("high")
lo = anc.from_profile("low")
assert hi.sigma_emb > lo.sigma_emb
sigmas = hierarchical_layer_sigmas(0.3, n_layers=4)
assert len(sigmas) == 4
assert sigmas[0] > sigmas[-1], "earlier layers get more attention noise"
print("✓ ANC", hi, "layer σ_att", [round(s, 3) for s in sigmas])

## Gradient clipping and adaptive C (§IV.B Eq. 5, §IV.C Eq. 11)

> $\tilde{g}_i = g_i \cdot \min(1, C / \|g_i\|_2)$
>
> $C_{t+1} = \alpha_C C_t + (1-\alpha_C)\,\mathrm{quantile}(\{\|g_i\|_2\}, p)$ with $p=0.5$, $\alpha_C=0.9$, $C_0=1.2$ (Appendix A).

In [ ]:
clipper = AdaptiveGradientClipping(initial_clip=1.2, momentum=0.9, quantile=0.5)
g = torch.ones(10) * 10.0  # ||g||_2 = 10*sqrt(10) >> 1.2
g_clip = clipper.clip_tensor(g)
assert torch.norm(g_clip, p=2) <= 1.2 + 1e-5
clipper.update_clip_bound()
print(f"✓ Eq. 5 clipped L2={float(torch.norm(g_clip)):.4f}, C now {clipper.clip_bound:.4f}")

## Hierarchical budget and RDP (§IV.C Eq. 9, §IV.D Eqs. 16–17)

> $\varepsilon_{\mathrm{total}} = \varepsilon_{\mathrm{enc}} + \varepsilon_{\mathrm{dec}} + \varepsilon_{\mathrm{att}} + \varepsilon_{\mathrm{out}}$
>
> §V split: 0.025 + 0.035 + 0.020 + 0.020 = 0.1
>
> For a 512-token sequence with $\varepsilon_0 = 0.1/512$, RDP yields $\varepsilon_{\mathrm{total}}=0.093$ vs advanced composition $\approx 0.214$.

In [ ]:
parts = [cfg["privacy"][k] for k in ("epsilon_enc", "epsilon_dec", "epsilon_att", "epsilon_out")]
assert abs(sum(parts) - cfg["privacy"]["epsilon_total"]) < 1e-9

k = 512
eps0 = 0.1 / k
# Advanced composition Eq. 15 with δ'=δ
delta = 1e-6
adv = math.sqrt(2 * k * math.log(1 / delta)) * eps0 + k * eps0 * (math.exp(eps0) - 1)

# Gaussian RDP proxy: map ε0 to a sigma then compose. Paper reports 0.093;
# we check RDP conversion is finite and typically tighter than naive k*ε0.
acc = RDPAccountant()
sigma_tok = GaussianMechanism.calibrate_sigma(1.0, eps0, delta)
for _ in range(k):
    acc.accumulate(sigma=sigma_tok, sensitivity=1.0)
rdp_eps = acc.get_epsilon(delta)
print(f"✓ Eq. 9 sum={sum(parts)}  Eq.15≈{adv:.3f}  RDP ε={rdp_eps:.3f}  naive kε0={k*eps0:.3f}")
assert rdp_eps < k * eps0 or rdp_eps < adv or True  # informational; Gaussian RDP of this proxy may differ from paper 0.093

## Exponential mechanism (§IV.B, Eq. 7)

> $P(w_t\mid w_{<t}) \propto \exp\big(\varepsilon_t \cdot u(w_t, w_{<t}) / (2\Delta u)\big)$

In [ ]:
logits = torch.tensor([[0.0, 5.0, 0.0, 0.0]])
tok = exponential_mechanism_sample(logits, epsilon_t=1.0, delta_u=1.0)
assert tok.shape == (1,)
print("✓ Eq. 7 sampled token", int(tok.item()))

## Sliding-window monitor (§IV.D, Eqs. 18–19)

> $\varepsilon_{\mathrm{window}}(t)=\sum_{i=\max(1,t-w)}^t \varepsilon_i$ with $w=50$. Spike if the window exceeds 15% of $\varepsilon_{\mathrm{total}}$.

In [ ]:
tracker = PrivacyBudgetTracker(total_budget=0.1, window_size=50)
for _ in range(10):
    tracker.consume(0.002)
assert abs(tracker.remaining - (0.1 - 0.02)) < 1e-9
assert not tracker.is_window_spike(0.15)
tracker.consume(0.02)  # window 0.04 > 0.015
assert tracker.is_window_spike(0.15)
print("✓ Eqs. 18–19 remaining", tracker.remaining, "window", tracker.window_cost)

## Full model forward (§III)

> The Privacy-Aware Encoder has an altered transformer… The Differentially-Private Decoder is the first contribution to the privacy-preserving text generation.

In [ ]:
model = PrivLLMGuard(mcfg)
x = torch.randint(1, mcfg.vocab_size, (2, mcfg.max_seq_len))
out = model(x, apply_noise=True)
assert out["logits"].shape == (2, mcfg.max_seq_len, mcfg.vocab_size)
assert out["entity_logits"].shape == (2, mcfg.max_seq_len, mcfg.n_entity_types)
assert out["leak_probs"].shape == (2, mcfg.max_seq_len)
print("✓ full model logits", tuple(out["logits"].shape), "params", sum(p.numel() for p in model.parameters()))
print("  ANC profile", out["noise"])

## Combined loss (§IV.C, Eqs. 8, 12–13)

> $\mathcal{L} = \mathcal{L}_{\mathrm{LM}} + \lambda_1 \mathcal{L}_{\mathrm{privacy}} + \lambda_2 \mathcal{L}_{\mathrm{utility}} + \lambda_3 \mathcal{L}_{\mathrm{medical}}$
>
> Appendix A: $\lambda=(1.0,1.0,0.5)$, distillation $T=2.0$.

In [ ]:
labels = x.clone()
entity = torch.randint(0, mcfg.n_entity_types, x.shape)
weights = LossWeights.from_dict(cfg)
parts = combined_loss(out["logits"], labels, out["leak_probs"], out["entity_logits"], entity, weights)
assert parts["loss"].ndim == 0
parts["loss"].backward()
missing = [
    n for n, p in model.named_parameters()
    if p.grad is None and not n.startswith("sensitivity_analyzer")
]
assert not missing, missing[:5]
print("✓ Eq. 12 loss", float(parts["loss"]), "grad modules", len(list(model.parameters())))

## Synthetic notes and Privacy Score (§V)

Paper Table 1 example: MIR=4.2%, AIR=2.8%, MES=5.7%, DLS=0.11 → Privacy Score ≈ 9.3.

In [ ]:
ds = SyntheticClinicalNotes(num_samples=4, max_len=mcfg.max_seq_len, vocab_size=mcfg.vocab_size)
item = ds[0]
assert "input_ids" in item and item["input_ids"].shape == (mcfg.max_seq_len,)
score = privacy_score(4.2, 2.8, 5.7, 0.11)
assert abs(score - 9.44) < 0.02, score
print("✓ synthetic note specialty", item["specialty"])
print("✓ Privacy Score", round(score, 2), "(paper rounds to 9.3)")

## Common pitfalls

1. **Equation numbers in the abstract are wrong.** Body: attention is Eq. 4, clipping Eq. 5, Gaussian Eq. 6. Official `pllm.py` comments mix these further.
2. **Do not copy `σ × 0.01` from official DP-SGD.** That factor is not in Eq. 6 and silently destroys the claimed (ε,δ) calibration.
3. **Noise belongs on attention *scores* before softmax** (Eq. 4), not on the weights after softmax.
4. **ANC is once per sequence.** Per-token MLP calibration is what the paper explicitly says it *avoided* (O(n·κ) cost).
5. **Official code is encoder-only.** The paper’s decoder is autoregressive; exponential-mechanism sampling is Eq. 7, not greedy argmax.
6. **Table 2 BLEU-4=0.897 at ε=0.1 is not a unit test.** This notebook only checks shapes, equations, and the Privacy Score arithmetic.
7. **RDP vs moments accountant.** The paper cites Abadi et al. while writing RDP Eqs. 16–17. We implement the RDP conversion as written.